[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/land-economics-ilr-uni-bonn/sat-agri-econ/blob/main/earth_engine/2026/02_research_bridge_notebooks/02_batch_export.ipynb)

# Exporting an Image

Move a result out of Earth Engine deliberately, and meet Earth Engine's pull-based projection model (scale, CRS, resampling).

## Setup



In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip install xee rioxarray


In [1]:
import ee
import geemap
import eerepr

In [2]:
cloud_project = 'ee-hadicu06'  # replace with your cloud project id

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

In [3]:
Map = geemap.Map()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Concept

Exports are the boundary between interactive exploration and batch computation. Small previews are synchronous; research-scale outputs should be explicit export tasks.

### Scaling pattern

Create export tasks deliberately, inspect their region/scale/maxPixels parameters, and start them only when you are confident. Large jobs may need tiling or chunking.

### Research use

Exported rasters are useful for reproducibility, archival, or downstream local processing. For econometrics, exports often become tables after zonal aggregation or sampling.

## Earth Engine's Projection Model (Pull-Based)

Traditional desktop GIS makes you align every layer to a common grid (extent, resolution, projection) *before* computing, and reprojecting/resampling is an expensive explicit step. Earth Engine inverts this: you usually specify the output grid only at the **end** of the chain -- through the `scale`/`crs`/`region` of an export, a `reduceRegion(s)` call, or simply the zoom and viewport of the map. Earth Engine then **pulls** inputs at exactly the tile and resolution needed and resamples/reprojects on the fly (it precomputes reduced-resolution image pyramids to do this efficiently).

Two practical consequences:

- The `scale` you pass to a reducer or export is a **measurement choice**, not a display setting: it sets the **pixel support** of your variable. A district mean at 250 m and at 1 km can differ, so choose scale deliberately.
- The default pyramiding (downsampling) policy is **mean** for continuous images. For **categorical** images that is wrong -- use nearest-neighbour / mode, or control resampling explicitly with `resample`, `reproject`, or `reduceResolution`.

**In short: the output grid is decided by the request at the end of the chain (the export or reducer), not fixed in advance as in desktop GIS.**

## Scaling: Interactive vs Batch, and How to Parallelize

Earth Engine has two execution modes. **Interactive** calls (`print`, map tiles, small `.getInfo()`) must finish in roughly five minutes with modest memory (tens of MB) -- use them only for small diagnostics. Research-scale work goes through **batch export tasks**, which run much longer (on the order of days; effectively up to 4 days) but allow only a few concurrent tasks (usually 2) per user.

Earth Engine automatically parallelizes `map()` and reducers across its servers; you **cannot tune that parallelism directly**, but you *can* shape the job so it fits the limits:

- **Filter first** (`filterBounds`, `filterDate`) so the server touches less data.
- **Chunk the work into many tasks** -- one export per country, per site, per year, or per variable -- and loop over them. This is the realistic parallelism pattern in research code.
- Raise **`tileScale`** (2, 4, 8, 16) when you hit *user memory limit exceeded* or *too many concurrent aggregations*; **simplify geometry** vertices when a job *times out*; split `reduceResolution` into two steps for *too many input pixels*; set a sane **`maxPixels`**.

Workflow habit: get it correct on **one small region or short window first**, then scale by submitting the chunked batch tasks -- never force a large result in interactive environment (e.g. .getInfo()).

## Geemap Workflow Pattern

Use geemap for notebook ergonomics: interactive maps, layer controls, legends, chart helpers, dataframe/export helpers, and drawn regions of interest. Keep raster algebra, reducers, masking, `ImageCollection.map()`, classifiers, and lazy server-side objects in the native Earth Engine `ee` API.

Geemap provides export helpers such as `geemap.ee_export_image_to_drive(...)`. This notebook keeps the native task object visible and does not start exports automatically; the geemap helper is shown as an optional pattern below.

In [ ]:
# Optional geemap export helper. Uncomment only when you intentionally want to start
# a Drive export. Keep the native task in the main code below to inspect export settings 
# before launching long-running jobs.
# geemap.ee_export_image_to_drive(
#     ndvi,
#     description="landsat_ndvi_example_geemap",
#     folder="earthengine",
#     region=roi,
#     scale=30,
# )


## One export job

In [4]:
geometry = ee.Geometry.Point([104.74551156654312, -2.9714700800585607])

# The export region (the source script's drawn polygon, here over Ethiopia)
aoi = ee.Geometry.Polygon(
    [[[38.07697234332294, 9.01840891655024],
      [38.10804305255146, 9.011118652896116],
      [38.13156066119404, 9.020612920770875],
      [38.11199126422138, 9.048077074209086],
      [38.06873259722919, 9.038583529531088]]])

In [ ]:
Map.addLayer(aoi)
Map.centerObject(aoi, zoom = 12)

Map


Map(bottom=7966373.0, center=[9.029016509312168, 38.09990534642131], controls=(WidgetControl(options=['positio…

In [10]:
# Build a cloud-masked Landsat-8 median composite over the AOI
l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
     .filterDate("2019-01-01", "2021-12-31") \
     .filter("CLOUD_COVER_LAND < 50") \
     .filterBounds(aoi)


def apply_scale_and_offset(image):
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = image.select("ST_B.*").multiply(0.00341802).add(149.0)
    return image.addBands(optical, None, True).addBands(thermal, None, True)


def mask_clouds(image):
    qa = image.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(clear)


composite = l8 \
            .map(apply_scale_and_offset) \
            .map(mask_clouds) \
            .median()

Map.addLayer(composite, 
{"min": 0, "max": 0.2, "bands": ["SR_B4", "SR_B3", "SR_B2"]}, 
"Median composite")

In [ ]:
# Keep reflective bands; scale x10000 and cast to int16 to shrink the file on disk
composite_refl_int = composite.select("SR_B.").multiply(10000).round().toInt16().set("scaling_factor", 10000)

In [12]:
composite_refl_int

In [ ]:
# --- Construct export tasks but do NOT auto-start them (inspect region/scale/CRS first) ---
# Replace the assetId below with your OWN asset path: 'users/<your-username>/...'.

# Export to Asset
task_asset = ee.batch.Export.image.toAsset(
    image=composite_refl_int,
    description="landsat_composite",
    assetId="users/hadicu06/tutorial_satAgEcon/landsat_composite",  # replace with your own asset id
    region=aoi,
    scale=30,
    crs="EPSG:4326",
    maxPixels=1e9)

task_asset.start()


In [ ]:
# Export to Drive

task_drive = ee.batch.Export.image.toDrive(
    image=composite_refl_int, 
    description="landsat_composite",
    folder="tutorial_satAgEcon", 
    region=aoi, 
    scale=30, 
    crs="EPSG:4326",
    maxPixels=1e9)
    
task_drive.start()

## Multiple export jobs

In [ ]:
# --- Exporting MULTIPLE images: one task per item, in a loop ---
# Batch exports are submitted as separate tasks, so to export many images you build a
# task per image and start each. Here: one annual median composite per year.
def annual_composite(year):
    start = ee.Date.fromYMD(year, 1, 1)
    yearly = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterDate(start, start.advance(1, "year"))
        .filter("CLOUD_COVER_LAND < 50")
        .filterBounds(aoi)
        .map(apply_scale_and_offset)
        .map(mask_clouds)
        .median()
        .select("SR_B.")
    )
    return yearly.set("year", year)

for year in [2019, 2020, 2021]:
    img = annual_composite(year)

    # --- Build an export task (do not auto-start; inspect region / scale / CRS first) ---
    task = ee.batch.Export.image.toDrive(
        image=img.multiply(10000).toInt16(),
        description="landsat_median_" + str(year),
        folder="tutorial_satAgEcon",
        region=aoi,
        scale=30,
        crs="EPSG:4326",
        maxPixels=1e9)

    task.start()  

### Tiling large exports


```python

# 1. Build a grid over your AOI
grid = largeMosaic.geometry().coveringGrid(...)  # or a custom FC
# 2. Keep only tiles that intersect the AOI
filtered_grid = grid.filterBounds(aoi)
# 3. Pull tile IDs to Python for looping + naming
tile_ids = filtered_grid.aggregate_array("system:index").getInfo()
# or some custom property like "tile_id"

# 4. Loop over tiles and export
for i, tile_id in enumerate(tile_ids):
    feature = ee.Feature(filtered_grid.toList(1, i).get(0))
    geometry = feature.geometry()
    task_name = 'tile_' + tile_id.replace(',', '_')
    task = ee.batch.Export.image.toDrive(**{
        'image': largeMosaic,
        'region': geometry,
        ...
    })
    task.start()

```